# 🐼 Panda AI — One-Click Deploy on Colab

Deploy the full Panda AI gateway (OpenAI-compatible API + Dashboard) in one click.

**What this does:**
1. Installs Python deps + Chromium headless + Node.js
2. Builds the Next.js dashboard
3. Starts the API server (port 8000) + Dashboard (port 5000)
4. Exposes both via cloudflared tunnels → **2 public URLs**

**Usage:** Runtime → Run all (Ctrl+F9)

After deploy:
1. Open the Dashboard link → paste your API token
2. Import cookies from your ChatGPT/Claude session
3. Use the API link as an OpenAI-compatible endpoint

> ⏱️ First run takes ~3 min. Sessions last ~12h on free Colab.

## 1️⃣ Install Dependencies

In [ ]:
# ── System deps (Chromium + Node.js) ──────────────────────────────────
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq nodejs npm > /dev/null 2>&1

# ── Python deps + Chromium (suppress warnings from pre-installed Colab packages) ──
!pip install -q -r requirements.txt --root-user-action=ignore 2>&1 | grep -v WARNING | tail -2
!patchright install chromium 2>&1 | tail -1

# ── Cloudflared (port tunnel) ─────────────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!mv /tmp/cloudflared /usr/local/bin/cloudflared 2>/dev/null || cp /tmp/cloudflared /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, sys
node_v = subprocess.check_output(['node', '--version']).decode().strip()
py_v = sys.version.split()[0]
print(f'✅ Python {py_v} | Node {node_v} | Chromium ready | cloudflared ready')

## 2️⃣ Clone & Configure

In [ ]:
import os, secrets

# ── Clean previous clone if exists ─────────────────────────────────────
!rm -rf /content/Panda-Ai

# ── Clone ──────────────────────────────────────────────────────────────
!git clone -q https://github.com/ferelking242/Panda-Ai.git /content/Panda-Ai
%cd /content/Panda-Ai

# ── Generate .env ──────────────────────────────────────────────────────
api_token = 'pnd_' + secrets.token_hex(16)
env_content = f"""
PROVIDER=chatgpt
HEADLESS=true
API_HOST=0.0.0.0
API_PORT=8000
API_TOKEN={api_token}
POOL_SIZE=1
RESPONSE_TIMEOUT=120000
LOG_LEVEL=INFO
""".strip()

with open('.env', 'w') as f:
    f.write(env_content)

print(f'✅ Repo cloned | Provider: chatgpt')
print(f'🔑 Token: {api_token}')

## 3️⃣ Build Dashboard

In [ ]:
%cd /content/Panda-Ai/dashboard
!npm install --no-audit --no-fund --registry=https://registry.npmjs.org/ --silent 2>&1 | tail -1
!npm run build --silent 2>&1 | tail -2
%cd /content/Panda-Ai
print('✅ Dashboard built')

## 4️⃣ Start Backend + Dashboard

In [ ]:
import subprocess, time, os, sys

# Kill any previous instances
!pkill -f 'uvicorn.*src.api.server' 2>/dev/null || true
!pkill -f 'node.*server.js' 2>/dev/null || true
time.sleep(1)

# ── Start API server ───────────────────────────────────────────────────
api_proc = subprocess.Popen(
    [sys.executable, '-m', 'src.api.server'],
    cwd='/content/Panda-Ai',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)
print(f'🚀 API server starting (PID {api_proc.pid})...')

# ── Start Dashboard ────────────────────────────────────────────────────
dash_proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='/content/Panda-Ai/dashboard',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'PORT': '5000', 'API_ORIGIN': 'http://127.0.0.1:8000', 'NODE_ENV': 'production'}
)
print(f'📊 Dashboard starting (PID {dash_proc.pid})...')

# ── Wait for health check ──────────────────────────────────────────────
import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3)
        print(f'✅ API healthy after {(i+1)*2}s')
        break
    except Exception:
        if i == 29:
            print('⚠️ API not yet healthy — check logs in next cell')

## 5️⃣ Expose Public URLs

In [ ]:
import re, threading, time

urls = {}

def start_tunnel(port, name):
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            urls[name] = match.group(0)
            print(f'  🔗 {name}: {match.group(0)}')
            break

t1 = threading.Thread(target=start_tunnel, args=(8000, 'API'))
t2 = threading.Thread(target=start_tunnel, args=(5000, 'Dashboard'))
t1.start()
t2.start()

# Wait for both tunnels
for _ in range(30):
    time.sleep(1)
    if len(urls) >= 2:
        break

print()
print('═' * 55)
print('  🐼 PANDA AI — DEPLOYED')
print('═' * 55)
if 'API' in urls:
    print(f'  🤖 API (OpenAI-compatible):')
    print(f'     {urls["API"]}/v1')
if 'Dashboard' in urls:
    print(f'  📊 Dashboard:')
    print(f'     {urls["Dashboard"]}')
print('═' * 55)
print(f'  🔑 Token: {api_token}')
print('═' * 55)
print()
print('Usage from any OpenAI client:')
print(f'  base_url = "{urls.get("API", "")}/v1"')
print(f'  api_key  = "{api_token}"')

## 📋 Quick Test

In [ ]:
# ── Test the API ───────────────────────────────────────────────────────
import urllib.request, json

if 'API' in urls:
    base = urls['API']
    
    # Health check
    health = json.loads(urllib.request.urlopen(f'{base}/healthz').read())
    print(f'Health: {health}')
    
    # List models
    req = urllib.request.Request(
        f'{base}/v1/models',
        headers={'Authorization': f'Bearer {api_token}'}
    )
    models = json.loads(urllib.request.urlopen(req).read())
    print(f'Models: {[m["id"] for m in models["data"][:5]]}')
    
    print('\n✅ Gateway is live and authenticated!')